# Create KWF Kankerbestrijding (Dutch Cancer Society) Awards

**KWF** (funder_id `4320322777`, ROR `0368jnd28`, present in common.funder,
priority `359`). Source: the funder's public Onderzoeksdatabase, served by a
**Solr** backend at `https://www.kwf.nl/odb_query` (POST Solr query, wt=json;
see scripts/local/kwf_to_s3.py). **1,003 projects, start-year 2017+**, native
project numbers, **100% title/PI/institution/start-date/abstract**, 98% modality.
PI titles ("dr. ...") stripped. NO per-project amount published (§6.7 waiver),
Dutch titles/abstracts. KWF had **~0 structured awards in OpenAlex** — high net-new.

Parquet: `s3://openalex-ingest/awards/kwf/kwf_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kwf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/kwf/kwf_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.kwf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.kwf_raw LIMIT 5;

## Step 2: Create KWF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kwf_awards
USING delta
AS
WITH
kwf_funder AS (
    -- KWF Kankerbestrijding F4320322777 in common.funder (ROR 0368jnd28, DOI 10.13039/501100004622)
    SELECT CAST(funder_id AS BIGINT) as funder_id, display_name, ror_id, doi
    FROM openalex.common.funder WHERE funder_id = 4320322777
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title), ''), CONCAT('KWF project ', g.funder_award_id)) as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,        -- KWF publishes no per-project amount (§6.7)
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,                    -- research modality (Immunotherapie, etc.)
        'kwf' as provenance,
        TRY_TO_DATE(g.start_date_raw, 'yyyy-MM-dd') as start_date,
        CAST(NULL AS DATE) as end_date,
        YEAR(TRY_TO_DATE(g.start_date_raw, 'yyyy-MM-dd')) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'Netherlands' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.kwf_raw g
    CROSS JOIN kwf_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'kwf' AND priority = 359;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    359 as priority
FROM openalex.awards.kwf_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(DISTINCT funder_award_id) uniq_award, COUNT(DISTINCT id) uniq_id,
  COUNT(DISTINCT funder_id) funders,
  SUM(CASE WHEN display_name IS NULL OR LENGTH(TRIM(display_name))=0 THEN 1 ELSE 0 END) blank,
  COUNT(lead_investigator) has_pi, COUNT(start_date) has_start, COUNT(funder_scheme) has_scheme,
  MIN(start_year) min_yr, MAX(start_year) max_yr
FROM openalex.awards.kwf_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) cnt
FROM openalex.awards.kwf_awards GROUP BY 1 ORDER BY 2 DESC LIMIT 12;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='kwf' AND priority=359;